In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()} if hasattr(model.config.id2label, "items") else dict(enumerate(model.config.id2label))
label_text = {i: str(v).lower() for i, v in id2label.items()}
entailment_id = next(i for i, v in label_text.items() if "entail" in v)
contradiction_id = next(i for i, v in label_text.items() if "contrad" in v)

print("model_name:", model_name)
print("id2label:", id2label)
print("entailment_id:", entailment_id, "contradiction_id:", contradiction_id)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model_name: typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
entailment_id: 0 contradiction_id: 2


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 64
preds = []
gap_12_all = []
gap_21_all = []
avg_gap_all = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc_12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc_21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc_12 = {k: v.to(device) for k, v in enc_12.items()}
        enc_21 = {k: v.to(device) for k, v in enc_21.items()}

        logits_12 = model(**enc_12).logits
        logits_21 = model(**enc_21).logits

        probs_12 = torch.softmax(logits_12, dim=-1)
        probs_21 = torch.softmax(logits_21, dim=-1)

        gap_12 = probs_12[:, entailment_id] - probs_12[:, contradiction_id]
        gap_21 = probs_21[:, entailment_id] - probs_21[:, contradiction_id]
        avg_gap = (gap_12 + gap_21) / 2.0

        batch_preds = (avg_gap > 0).long().cpu().numpy()

        preds.extend(batch_preds.tolist())
        gap_12_all.extend(gap_12.cpu().numpy().tolist())
        gap_21_all.extend(gap_21.cpu().numpy().tolist())
        avg_gap_all.extend(avg_gap.cpu().numpy().tolist())

y_pred = np.array(preds)
gap_12_all = np.array(gap_12_all)
gap_21_all = np.array(gap_21_all)
avg_gap_all = np.array(avg_gap_all)

print("done")

  0%|          | 0/7 [00:00<?, ?it/s]

done


In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.7181372549019608, 'f1': 0.7905282331511839}
                precision    recall  f1-score   support

not_paraphrase       0.55      0.59      0.57       129
    paraphrase       0.80      0.78      0.79       279

      accuracy                           0.72       408
     macro avg       0.68      0.68      0.68       408
  weighted avg       0.72      0.72      0.72       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("gap_12:", float(gap_12_all[i]))
    print("gap_21:", float(gap_21_all[i]))
    print("avg_gap:", float(avg_gap_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
gap_12: 0.9963528513908386
gap_21: 0.8122128844261169
avg_gap: 0.9042828679084778
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
gap_12: -0.9993698596954346
gap_21: -0.000674131908454001
avg_gap: -0.5000219941139221
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
gap_12: -0.00016976663027890027
gap_21: 0.9945570826530457
avg_gap: 0.49719

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("gap_12:", float(gap_12_all[i]))
    print("gap_21:", float(gap_21_all[i]))
    print("avg_gap:", float(avg_gap_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

num_errors: 115
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
gap_12: -0.00016976663027890027
gap_21: 0.9945570826530457
avg_gap: 0.49719366431236267
true: 0 pred: 1
idx: 4
sentence1: No dates have been set for the civil or the criminal trial .
sentence2: No dates have been set for the criminal or civil cases , but Shanley has pleaded not guilty .
gap_12: -0.0002203288022428751
gap_21: 0.9956680536270142
avg_gap: 0.49772384762763977
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
gap_12: -0.00019033583521377295
gap_21: -0.0004739669

In [8]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
    "method": "mean_bidirectional_softmax_gap",
}
summary

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.7181372549019608,
 'f1': 0.7905282331511839,
 'method': 'mean_bidirectional_softmax_gap'}